# RajeshGPT — train on Colab

Before running anything: **Runtime -> Change runtime type -> T4 GPU** (free tier), then Save.

This notebook is self-contained. Run the cells top to bottom.

## 1. Get the training data (Tiny Shakespeare)

In [1]:
!wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!wc -l input.txt

40000 input.txt


## 2. Tokenizer + data loading

In [2]:
import torch

class CharTokenizer:
    def __init__(self, text):
        chars = sorted(set(text))
        self.vocab_size = len(chars)
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)


def load_data(path, val_fraction=0.1):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    tok = CharTokenizer(text)
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    n = int(len(data) * (1 - val_fraction))
    return data[:n], data[n:], tok


def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## 3. The model

In [3]:
import math
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class RajeshGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=128, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, "sequence longer than block_size"
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

## 4. Train

Check the printout says `device: cuda`. If it says `cpu`, set the runtime type (top of notebook) and re-run from the top.

In [4]:
import time

BLOCK_SIZE = 128
BATCH_SIZE = 64
N_EMBD = 192
N_HEAD = 6
N_LAYER = 6
DROPOUT = 0.1
LR = 3e-4
MAX_ITERS = 5000
EVAL_INTERVAL = 250
EVAL_ITERS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 1337

torch.manual_seed(SEED)
print(f"device: {DEVICE}")

train_data, val_data, tok = load_data("input.txt")
print(f"vocab_size={tok.vocab_size}  train_tokens={len(train_data)}  val_tokens={len(val_data)}")

model = RajeshGPT(
    vocab_size=tok.vocab_size,
    block_size=BLOCK_SIZE,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_layer=N_LAYER,
    dropout=DROPOUT,
).to(DEVICE)
print(f"model params: {model.num_params():,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_ITERS)


@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split, data in [("train", train_data), ("val", val_data)]:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            x, y = get_batch(data, BLOCK_SIZE, BATCH_SIZE, DEVICE)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


t0 = time.time()
for it in range(1, MAX_ITERS + 1):
    x, y = get_batch(train_data, BLOCK_SIZE, BATCH_SIZE, DEVICE)
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if it % EVAL_INTERVAL == 0 or it == 1:
        losses = estimate_loss()
        elapsed = time.time() - t0
        print(f"iter {it:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | {elapsed:.1f}s elapsed")

torch.save(
    {
        "model_state": model.state_dict(),
        "config": {
            "vocab_size": tok.vocab_size,
            "block_size": BLOCK_SIZE,
            "n_embd": N_EMBD,
            "n_head": N_HEAD,
            "n_layer": N_LAYER,
            "dropout": DROPOUT,
        },
        "stoi": tok.stoi,
        "itos": tok.itos,
    },
    "rajeshgpt.pt",
)
print("saved checkpoint to rajeshgpt.pt")

device: cuda
vocab_size=65  train_tokens=1003854  val_tokens=111540
model params: 2,719,104
iter     1 | train loss 3.7483 | val loss 3.7644 | 3.6s elapsed
iter   250 | train loss 2.3002 | val loss 2.3153 | 24.5s elapsed
iter   500 | train loss 2.0051 | val loss 2.0717 | 46.2s elapsed
iter   750 | train loss 1.7985 | val loss 1.9290 | 68.2s elapsed
iter  1000 | train loss 1.6593 | val loss 1.8293 | 91.1s elapsed
iter  1250 | train loss 1.5681 | val loss 1.7364 | 115.6s elapsed
iter  1500 | train loss 1.4985 | val loss 1.6841 | 140.2s elapsed
iter  1750 | train loss 1.4563 | val loss 1.6412 | 163.7s elapsed
iter  2000 | train loss 1.4111 | val loss 1.6158 | 187.2s elapsed
iter  2250 | train loss 1.3854 | val loss 1.5926 | 211.3s elapsed
iter  2500 | train loss 1.3627 | val loss 1.5719 | 235.3s elapsed
iter  2750 | train loss 1.3453 | val loss 1.5562 | 259.1s elapsed
iter  3000 | train loss 1.3273 | val loss 1.5451 | 282.9s elapsed
iter  3250 | train loss 1.3101 | val loss 1.5394 | 307.0

## 5. Generate text from the trained model

In [5]:
def sample(prompt="ROMEO:", n_tokens=300, temperature=0.8, top_k=40):
    model.eval()
    idx = torch.tensor([[tok.stoi[c] for c in prompt]], dtype=torch.long, device=DEVICE)
    out = model.generate(idx, max_new_tokens=n_tokens, temperature=temperature, top_k=top_k)
    print(tok.decode(out[0].tolist()))

sample("ROMEO:", n_tokens=400)

ROMEO:
Why, receive on my life, but in the men:
There's times that then did set beat their deputy.

ESCALUS:
By my lord; and let us it more but his antrease:
Why, thou one that I see his true: indeeds,
We not say ever say I be a long; and so recontent
To ripen thee, and the grave not longer that my use
draw my grace of thy free traitor: all thy love.
I am confess to the city's of the world,
When they ar


## 6. Download the checkpoint to use locally with the website

Drop the downloaded `rajeshgpt.pt` into your local `rajeshgpt` project folder (replacing the old one), then run `python3 serve.py` and open `index.html` as before.

In [6]:
from google.colab import files
files.download("rajeshgpt.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>